In [85]:
import polars as pl

In [86]:
tournament_slug = "ngpr"

In [93]:
id_to_name = pl.DataFrame([
    # "Shane Fuller",
    # "Mason Hoppe",
    # "Liam Healy",
    # "Colin Allen",
    # "Nico Tapiero",
    (739817, "Joshua Tamayo"),
    (655745, "Martin Kessler"),
    (717730, "John Adam Rodriguez"),
    (543413, "Edy Salcedo"),
    (668760, "Benjamin Brown"),
    (3048904, "Trevor Swartz"),
    (1014980, "Myles Vigil"),
    (356016, "Cameron Garrison"),
    (1989880, "Morgan Bruce"),
    (2991441, "Maevey Kennedy"),
    (184972, "Danny Edlin"),
    (187, "Dan Cushing"),
    (7545, "Anthony Amaral"),
    (618047, "Alp Kuleli"),
    (45009, "Nico Tapiero"),
    (34096, "Jake Donovan"),
    (304, "Travis Flynn"),
], orient='row', schema=['global_id', 'venmo_name'])

In [3]:
import os
from google_auth_oauthlib.flow import InstalledAppFlow
import polars as pl

# Define the scopes required for Gmail access
SCOPES = [
    'https://www.googleapis.com/auth/gmail.readonly',
]

client_secret_path = 'client_secret.json'  # Update this path if your file is elsewhere

# Run the OAuth flow to get credentials
flow = InstalledAppFlow.from_client_secrets_file(
    client_secret_path, SCOPES)
creds = flow.run_local_server(port=0)

# Access token
access_token = creds.token

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=848371485907-cc7icscjoiemnvthija9q4jgut8oj69p.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A44795%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=1FQvvyanSBwdPOwVdCT2w0WqwpJNRp&access_type=offline


Error: Failed to open Wayland display, fallback to X11. WAYLAND_DISPLAY='wayland-0' DISPLAY=':0'


In [4]:
from googleapiclient.discovery import build
from datetime import datetime, timedelta
import email
import base64

# Build the Gmail API service
service = build('gmail', 'v1', credentials=creds)

# Get today's date in RFC 3339 format (YYYY/MM/DD)
today = datetime.now().strftime('%Y/%m/%d')
today = (datetime.now() - timedelta(days=7)).strftime('%Y/%m/%d')

# Search query for emails from venmo@venmo.com received today
query = f'from:venmo@venmo.com after:{today}'

# Fetch messages
results = service.users().messages().list(userId='me', q=query).execute()
messages = results.get('messages', [])

print(f"Found {len(messages)} emails from venmo@venmo.com today.")

# Get all messages and convert them to Python email.message.Message objects
email_messages = []
if messages:
    for msg_meta in messages:
        msg_id = msg_meta['id']
        msg = service.users().messages().get(userId='me', id=msg_id, format='raw').execute()
        raw_msg = base64.urlsafe_b64decode(msg['raw'].encode('ASCII'))
        email_message = email.message_from_bytes(raw_msg)
        email_messages.append(email_message)
    print(f"Converted {len(email_messages)} messages to email.message.Message objects.")
else:
    print("No emails found from venmo@venmo.com today.")

Found 24 emails from venmo@venmo.com today.
Converted 24 messages to email.message.Message objects.


In [5]:
from bs4 import BeautifulSoup

def extract_venmo_details(email_message):
    for part in email_message.walk():
        if part.get_content_type() == "text/html":
            html_body = part.get_payload(decode=True).decode('utf-8')
            soup = BeautifulSoup(html_body, 'html.parser')
            # Example: Find the amount, sender, and date
            # You must update these selectors based on the actual HTML structure
            amount = None
            sender = None
            date = None

            try:
                # Example extraction (update as needed)
                amount_tag = soup.find(string=lambda text: text and "$" in text)
                if amount_tag:
                    amount = amount_tag.strip().split("$")[1].split()[0]
                else:
                    return None
            except Exception as e:
                return None

            try:
                sender_tag = soup.find(string=lambda text: text and " paid you " in text)
                if sender_tag:
                    sender = sender_tag.text.split(' paid you ')[0].strip()
                else:
                    return None
            except Exception as e:
                return None

            return {
                "amount": amount,
                "sender": sender,
            }
    return None

In [6]:
payment_details = [
    extract_venmo_details(email_message)
    for email_message in email_messages
    if extract_venmo_details(email_message) is not None
]
payment_details

[{'amount': '10.00', 'sender': 'John Adam Rodriguez'},
 {'amount': '10.00', 'sender': 'Martin Kessler'},
 {'amount': '10.00', 'sender': 'Shane Fuller'},
 {'amount': '10.00', 'sender': 'Alp Kuleli'},
 {'amount': '10.00', 'sender': 'Maevey Kennedy'},
 {'amount': '10.00', 'sender': 'Mason Hoppe'},
 {'amount': '10.00', 'sender': 'Edy Salcedo'},
 {'amount': '10.00', 'sender': 'Benjamin Brown'},
 {'amount': '10.00', 'sender': 'Liam Healy'},
 {'amount': '10.00', 'sender': 'Trevor Swartz'},
 {'amount': '10.00', 'sender': 'Myles Vigil'},
 {'amount': '10.00', 'sender': 'Anthony Amaral'},
 {'amount': '10.00', 'sender': 'Travis Flynn'},
 {'amount': '10.00', 'sender': 'Colin Allen'},
 {'amount': '10.00', 'sender': 'Nico Tapiero'},
 {'amount': '10.00', 'sender': 'Joshua Tamayo'},
 {'amount': '10.00', 'sender': 'Jake Donovan'},
 {'amount': '10.00', 'sender': 'Cameron Garrison'},
 {'amount': '10.00', 'sender': 'Morgan Bruce'},
 {'amount': '10.00', 'sender': 'Danny Edlin'}]

In [53]:
payments = pl.DataFrame(payment_details, schema_overrides={
    "amount": pl.Float64,})

# Mark all as paid

In [7]:
import json
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

In [8]:
import os

auth_token = os.environ['SMASHGG_TOKEN']
api_version = 'alpha'

In [9]:
transport = RequestsHTTPTransport(
    url=f'https://api.start.gg/gql/{api_version}',
    headers={'Authorization': f'Bearer {auth_token}'},
    use_json=True,
)

client = Client(transport=transport, fetch_schema_from_transport=False)


In [ ]:
# Define the API endpoint and query for tournament details
query = gql("""
query TournamentQuery($slug: String!) {
    tournament(slug: $slug) {
        id
        name
        city
        state
        countryCode
        startAt
        endAt
        events {
            id
            name
            numEntrants
        }
    }
}
""")

variables = {"slug": tournament_slug}

# Execute the query
try:
        tournament_result = client.execute(query, variable_values=variables)
except Exception as e:
        print("Error fetching tournament details:", e)

{
  "tournament": {
    "id": 808121,
    "name": "New Game Plus Revival 8.13",
    "city": "Boston",
    "state": 3,
    "countryCode": "US",
    "startAt": 1752616800,
    "endAt": 1752638340,
    "events": [
      {
        "id": 1417057,
        "name": "Project+ Singles (Free!)",
        "numEntrants": 1
      },
      {
        "id": 1417055,
        "name": "Melee Singles",
        "numEntrants": 30
      },
      {
        "id": 1417056,
        "name": "Melee Redemption",
        "numEntrants": 9
      }
    ]
  }
}


In [ ]:
# Query to get all attendees (entrants) for the tournament using its id
attendees_query = gql("""
query TournamentAttendees($tournamentId: ID!) {
    tournament(id: $tournamentId) {
        participants(query: {page: 1, perPage: 512}) {
            nodes {
                id
                gamerTag
                user {
                    id
                    name
                }
            }
        }
    }
}
""")

attendees_variables = {"tournamentId": tournament_result['tournament']['id']}

try:
        result = client.execute(attendees_query, variable_values=attendees_variables)
        attendees_result = result['tournament']['participants']['nodes']
except Exception as e:
        print("Error fetching attendees:", e)

[
  {
    "id": 19052394,
    "gamerTag": "Pizza",
    "user": {
      "id": 184972,
      "name": "Daniel Edlin"
    }
  },
  {
    "id": 19051640,
    "gamerTag": "Skynative",
    "user": {
      "id": 173855,
      "name": null
    }
  },
  {
    "id": 19051485,
    "gamerTag": "nilo",
    "user": {
      "id": 1085569,
      "name": null
    }
  },
  {
    "id": 19040640,
    "gamerTag": "Ant",
    "user": {
      "id": 7545,
      "name": "Anthony Amaral"
    }
  },
  {
    "id": 19039985,
    "gamerTag": "bfu",
    "user": {
      "id": 2900434,
      "name": null
    }
  },
  {
    "id": 19053159,
    "gamerTag": "MAIF",
    "user": {
      "id": 2991441,
      "name": null
    }
  },
  {
    "id": 19024106,
    "gamerTag": "zaubermaus",
    "user": {
      "id": 305778,
      "name": "Aaron Giera"
    }
  },
  {
    "id": 19041987,
    "gamerTag": "yungsos",
    "user": {
      "id": 1004953,
      "name": null
    }
  },
  {
    "id": 19050801,
    "gamerTag": "Qwerty",
    "u

In [ ]:
data = []
for entrant in attendees_result:
    entrant_id = entrant['id']
    gamer_tag = entrant['gamerTag']
    entrant_name = entrant['user']['name'] if entrant['user'] else 'Unknown'
    global_id = entrant['user']['id'] if entrant['user'] else None
    data.append({'global_id': global_id, 'id': entrant_id, 'name': entrant_name, 'gamertag': gamer_tag})

attendees = pl.DataFrame(data)

global_id,id,name,gamertag
i64,i64,str,str
184972,19052394,"""Daniel Edlin""","""Pizza"""
173855,19051640,null,"""Skynative"""
1085569,19051485,null,"""nilo"""
7545,19040640,"""Anthony Amaral""","""Ant"""
2900434,19039985,null,"""bfu"""
2991441,19053159,null,"""MAIF"""
305778,19024106,"""Aaron Giera""","""zaubermaus"""
1004953,19041987,null,"""yungsos"""
107947,19050801,"""Lucas Prosperino""","""Qwerty"""


## get entrants for each event

In [21]:
def get_event_id(event_name):
    # Extract the event id for a given event name
    events = tournament_result['tournament']['events']
    event_id = next((event['id'] for event in events if event_name in event['name']), None)
    return event_id

In [31]:
import polars as pl

# Extract entrant data and flatten participants' gamerTags

def pl_from_players_json(players_json):
    data = []
    for entrant in players_json:
        entrant_id = entrant['id']
        entrant_name = entrant['name']
        # There may be multiple participants per entrant; join their gamerTags with comma
        gamer_tags = [p['gamerTag'] for p in entrant.get('participants', [])]
        gamer_tag = ', '.join(gamer_tags)
        data.append({'id': entrant_id, 'name': entrant_name, 'gamertag': gamer_tag})

    return pl.DataFrame(data)

In [32]:
def get_entrants(event_name):
    # Get entrants for a specific event by name
    event_id = get_event_id(event_name)
    if not event_id:
        print(f"Event '{event_name}' not found.")
        return []

    entrants_query = gql("""
    query EventEntrants($eventId: ID!) {
        event(id: $eventId) {
            entrants(query: {page: 1, perPage: 512}) {
                nodes {
                    id
                    name
                    participants {
                        gamerTag
                    }
                }
            }
        }
    }
    """)

    entrants_variables = {"eventId": event_id}

    try:
        result = client.execute(entrants_query, variable_values=entrants_variables)
        return pl_from_players_json(result['event']['entrants']['nodes'])
    except Exception as e:
        print(f"Error fetching entrants for {event_name}:", e)
        return None

In [33]:
melee_singles = get_entrants("Melee Singles")

In [34]:
p_plus = get_entrants("Project+ Singles")

In [110]:
pl.Config(tbl_rows=50)

payment_status = (
    attendees
    .join(
        melee_singles.with_columns(pl.lit(True).alias('melee')),
        on='gamertag', how='left', suffix='_melee',
    )
    .join(
        p_plus.with_columns(pl.lit(True).alias('p_plus')),
        on='gamertag', how='left', suffix='_p_plus',
    )
    .join(attendees, on='id', how='left')
    .join(
        (
            payments
            .join(id_to_name, left_on='sender', right_on='venmo_name', how='left')
            .with_columns(pl.lit(True).alias('paid'))
        ),
        on='global_id', how='left', suffix='_paid',
    )
    .fill_null(False)
    .select('global_id', 'id', 'name', 'gamertag', 'melee', 'p_plus', 'paid', 'amount')
)
payment_status.select(pl.exclude('^.*id$'))

name,gamertag,melee,p_plus,amount
str,str,bool,bool,f64
"""Daniel Edlin""","""Pizza""",true,false,10.0
null,"""Skynative""",true,false,null
null,"""nilo""",true,false,null
"""Anthony Amaral""","""Ant""",true,false,10.0
null,"""bfu""",true,false,null
null,"""MAIF""",true,false,10.0
"""Aaron Giera""","""zaubermaus""",true,false,null
null,"""yungsos""",true,false,null
"""Lucas Prosperino""","""Qwerty""",true,false,null


# mark ettendees as paid

In [ ]:
paid_tounament_ids = (
    payment_status
    .filter('paid')
    .filter('melee')
    .select('global_id')
    .to_series()
    .to_list()
)

In [104]:
event_id = get_event_id("Melee Singles")

In [108]:
# Register all paid tournament IDs for the Melee Singles event

registration_input = {
    "eventIds": [event_id]  # event_id is for "Melee Singles"
}

for entrant_id in paid_tounament_ids:
    # 1. Generate registration token
    generate_token_mutation = gql("""
    mutation GenerateRegistrationToken($registration: TournamentRegistrationInput!, $userId: ID!) {
        generateRegistrationToken(registration: $registration, userId: $userId)
    }
    """)
    variables = {
        "registration": registration_input,
        "userId": entrant_id
    }
    try:
        token_result = client.execute(generate_token_mutation, variable_values=variables)
        registration_token = token_result["generateRegistrationToken"]
    except Exception as e:
        print(f"Error generating registration token for entrant {entrant_id}: {e}")
        continue

    # 2. Register for tournament
    register_mutation = gql("""
    mutation RegisterForTournament($registration: TournamentRegistrationInput!, $registrationToken: String!) {
        registerForTournament(registration: $registration, registrationToken: $registrationToken) {
            id
        }
    }
    """)
    variables = {
        "registration": registration_input,
        "registrationToken": registration_token
    }
    try:
        reg_result = client.execute(register_mutation, variable_values=variables)
        print(f"Registered entrant {entrant_id} for event {event_id}: {reg_result}")
    except Exception as e:
        print(f"Error registering entrant {entrant_id}: {e}")

Registered entrant 305778 for event 1417055: {'registerForTournament': {'id': 19024106}}
